<a href="https://colab.research.google.com/github/blbl-blbl/study/blob/main/PyTorch/01_oxford_pets/08_test_and_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oxford-IIIT Pet — Test & Inference

Этот notebook полностью автономен: он содержит собственную подготовку данных и модели, финальную оценку на `test` и inference. Запуск предыдущих notebooks не требуется.

## Самостоятельная подготовка модели

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

raw_dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    download=True,
)

labels = np.array([raw_dataset[i][1] for i in range(len(raw_dataset))])

train_indices, val_indices = train_test_split(
    np.arange(len(raw_dataset)),
    test_size=0.2,
    random_state=42,
    stratify=labels,
)

class_names = raw_dataset.classes
weights = ResNet18_Weights.IMAGENET1K_V1
resnet_transform = weights.transforms()

resnet_train_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_val_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_transform,
    download=False,
)

resnet_train_dataset = Subset(resnet_train_source, train_indices.tolist())
resnet_val_dataset = Subset(resnet_val_source, val_indices.tolist())

resnet_train_loader = DataLoader(
    resnet_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

resnet_val_loader = DataLoader(
    resnet_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

loss_fn = nn.CrossEntropyLoss()

print("Train:", len(resnet_train_dataset))
print("Validation:", len(resnet_val_dataset))
print("Classes:", len(class_names))

In [ ]:
# Функция обучения
def train_head_one_epoch(
    model, dataloader, loss_fn, optimizer, device
):
  model.eval()
  model.fc.train()

  total_loss = 0
  total_correct = 0
  total_objects = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    logits = model(images)
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)
    total_loss += loss.item() * batch_size
    total_correct += (
        logits.argmax(dim=1) == labels
    ).sum().item()
    total_objects += batch_size

  return (
      total_loss / total_objects,
      total_correct / total_objects
  )


  # Функция оценки
def evaluate_metrics(model, dataloader, loss_fn, device, num_classes):
  model.eval()

  total_loss = 0.0
  total_correct = 0
  total_top3_correct = 0
  total_objects = 0

  all_labels = []
  all_predictions = []

  with torch.inference_mode():
      for images, labels in dataloader:
          images = images.to(device)
          labels = labels.to(device)

          logits = model(images)
          loss = loss_fn(logits, labels)

          predictions = logits.argmax(dim=1)
          top3 = logits.topk(k=3, dim=1).indices

          batch_size = labels.size(0)

          total_loss += loss.item() * batch_size
          total_correct += (predictions == labels).sum().item()
          total_top3_correct += (
              (top3 == labels.unsqueeze(1))
              .any(dim=1)
              .sum()
              .item()
          )
          total_objects += batch_size

          all_labels.extend(labels.cpu().tolist())
          all_predictions.extend(predictions.cpu().tolist())

  macro_f1 = f1_score(
      all_labels,
      all_predictions,
      labels=list(range(num_classes)),
      average="macro",
      zero_division=0,
  )

  return {
      "loss": total_loss / total_objects,
      "accuracy": total_correct / total_objects,
      "macro_f1": macro_f1,
      "top3_accuracy": total_top3_correct / total_objects,
  }

In [ ]:
# Мягкая аугментация (только для train)
resnet_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Validation-преобразования оставим прежними и детерменированными
resnet_val_transform = weights.transforms()


aug_train_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_train_transform,
    download=False,
)

aug_val_source = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=resnet_val_transform,
    download=False,
)


aug_train_dataset = Subset(
    aug_train_source,
    train_indices.tolist()
)

aug_val_dataset = Subset(
    aug_val_source,
    val_indices.tolist()
)


aug_train_loader = DataLoader(
    aug_train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42)
)

aug_val_loader = DataLoader(
    aug_val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Функция обучения для layer4 и fc
def train_layer4_one_epoch(
    model, dataloader, loss_fn, optimizer, device
):
  # Замороженная часть остается в eval
  model.eval()

  # Дообучаемые части переводим в train
  model.fc.train()
  model.layer4.train()

  total_loss = 0.0
  total_correct = 0
  total_objects = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    logits = model(images)
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()

    batch_size = labels.size(0)

    total_loss += loss.item() * batch_size
    total_correct += (
        logits.argmax(dim=1) == labels
    ).sum().item()
    total_objects += batch_size

  return (
      total_loss / total_objects,
      total_correct / total_objects
  )

In [ ]:
# Базовая модель
seed_everything(42)
resnet_train_loader.generator.manual_seed(42)

resnet_model = resnet18(weights=weights)

for parameter in resnet_model.parameters():
  parameter.requires_grad = False

resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    len(class_names),
)

resnet_model = resnet_model.to(device)

# Передаем оптимизатору только параметры последнего слоя
resnet_optimizer = torch.optim.Adam(
    resnet_model.fc.parameters(),
    lr=0.001,
)
loss_fn = nn.CrossEntropyLoss()

resnet_checkpoint_path = Path(
    "checkpoints/resnet18_head.pth"
)

resnet_checkpoint_path.parent.mkdir(
    parents=True, exist_ok=True
)

resnet_history = []

best_macro_f1 = -float("inf")
best_epoch = None
epochs_without_improvement = 0

max_epochs = 15
patience = 3
min_delta = 1e-4

for epoch in range(1, max_epochs+1):
  train_loss, train_accuracy = train_head_one_epoch(
      resnet_model,
      resnet_train_loader,
      loss_fn,
      resnet_optimizer,
      device,
  )

  metrics = evaluate_metrics(
      resnet_model,
      resnet_val_loader,
      loss_fn,
      device,
      num_classes=len(class_names),
  )

  resnet_history.append({
      'epoch': epoch,
      'train_loss': train_loss,
      'train_accuracy': train_accuracy,
      'val_loss': metrics["loss"],
      'val_accuracy': metrics['accuracy'],
      'val_macro_f1': metrics['macro_f1'],
      'val_top3_accuracy': metrics['top3_accuracy'],
  })

  status = ""

  if metrics['macro_f1'] > best_macro_f1 + min_delta:
    best_macro_f1 = metrics['macro_f1']
    best_epoch = epoch
    epochs_without_improvement = 0

    torch.save({
        'model_state_dict': resnet_model.state_dict(),
        'epoch': epoch,
        'metrics': metrics,
        'class_names': class_names,
        'weights': 'IMAGENET1K_V1',
    }, resnet_checkpoint_path)

    status = 'saved'

  else:
    epochs_without_improvement += 1
    status = (
        f"No improvement: "
        f"{epochs_without_improvement}/{patience}"
    )

  print(
      f"Epoch {epoch}/{max_epochs} | "
      f"Train loss: {train_loss:.4f} | "
      f"Train accuracy: {train_accuracy:.2%} | "
      f"Val loss: {metrics['loss']:.4f} | "
      f"Val accuracy: {metrics['accuracy']:.2%} | "
      f"Val macro-F1: {metrics['macro_f1']:.4f} | "
      f"{status}"
  )

  if epochs_without_improvement >= patience:
    print("Early stopping")
    break

In [ ]:
seed_everything(42)
aug_train_loader.generator.manual_seed(42)

head_checkpoint = torch.load(
    resnet_checkpoint_path,
    map_location='cpu',
    weights_only=True,
)

aug_fine_tune_model = resnet18(weights=None)

aug_fine_tune_model.fc = nn.Linear(
    aug_fine_tune_model.fc.in_features,
    len(class_names)
)

aug_fine_tune_model.load_state_dict(
    head_checkpoint["model_state_dict"]
)

for parameter in aug_fine_tune_model.parameters():
  parameter.requires_grad=False

for parameter in aug_fine_tune_model.layer4.parameters():
  parameter.requires_grad=True

for parameter in aug_fine_tune_model.fc.parameters():
  parameter.requires_grad=True


aug_fine_tune_model = aug_fine_tune_model.to(device)

aug_fine_tune_optimizer = torch.optim.Adam([
    {
        "params": aug_fine_tune_model.layer4.parameters(),
        "lr": 1e-5,
    },
    {
        "params": aug_fine_tune_model.fc.parameters(),
        "lr": 1e-4,
    }
])


In [ ]:
aug_fine_tune_checkpoint_path = Path(
    "checkpoints/resnet18_layer4_augmented.pth"
)

aug_fine_tune_history = []

best_macro_f1 = head_checkpoint["metrics"]["macro_f1"]
best_epoch = 0
epochs_without_improvement = 0

max_epochs = 15
patience = 3
min_delta = 1e-4

for epoch in range(1, max_epochs+1):
  train_loss, train_accuracy = train_head_one_epoch(
      aug_fine_tune_model,
      aug_train_loader,
      loss_fn,
      aug_fine_tune_optimizer,
      device,
  )

  metrics = evaluate_metrics(
      aug_fine_tune_model,
      aug_val_loader,
      loss_fn,
      device,
      num_classes=len(class_names),
  )

  aug_fine_tune_history.append({
      "epoch": epoch,
      "train_loss": train_loss,
      "train_accuracy": train_accuracy,
      "val_loss": metrics["loss"],
      "val_accuracy": metrics["accuracy"],
      "val_macro_f1": metrics["macro_f1"],
      "val_top3_accuracy": metrics["top3_accuracy"],
  })

  if metrics['macro_f1'] > best_macro_f1 + min_delta:
    best_macro_f1 = metrics['macro_f1']
    best_epoch = epoch
    epochs_without_improvement = 0
    status = 'saved'

    torch.save(
        {
            "model_state_dict": aug_fine_tune_model.state_dict(),
            "optimizer_state_dict": aug_fine_tune_optimizer.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "class_names": class_names,
            "weights": "IMAGENET1K_V1",
            "stage": "layer4_fc_fine_tuning",
        },
        aug_fine_tune_checkpoint_path
    )

  else:
      epochs_without_improvement += 1
      status = (
          f"No improvement: "
          f"{epochs_without_improvement}/{patience}"
      )

  print(
      f"Epoch {epoch}/{max_epochs} | "
      f"Train loss: {train_loss:.4f} | "
      f"Train accuracy: {train_accuracy:.2%} | "
      f"Val loss: {metrics['loss']:.4f} | "
      f"Val accuracy: {metrics['accuracy']:.2%} | "
      f"Val macro-F1: {metrics['macro_f1']:.4f} | "
      f"{status}"
  )

  if epochs_without_improvement >= patience:
      print("Early stopping")
      break

print("Best F1-macro:", best_macro_f1)

## Финальная оценка на `test`

In [15]:
test_dataset = OxfordIIITPet(
    root="data",
    split="test",
    target_types="category",
    transform=resnet_val_transform,
    download=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

checkpoint = torch.load(
    aug_fine_tune_checkpoint_path,
    map_location=device,
    weights_only=True
)

assert list(test_dataset.classes) == list(
    checkpoint["class_names"]
)

aug_fine_tune_model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_metrics = evaluate_metrics(
    aug_fine_tune_model,
    test_loader,
    loss_fn,
    device,
    num_classes=len(class_names)
)


print("Checkpoint epoch:", checkpoint["epoch"])
print("Test images:", len(test_dataset))

for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}")

Checkpoint epoch: 1
Test images: 3669
loss: 0.3270
accuracy: 0.8953
macro_f1: 0.8949
top3_accuracy: 0.9864


Финальная test-выборка используется только после выбора модели и завершения всех экспериментов на train/validation.

## Инференс

Функция получает путь к фотографии и возвращает три наиболее вероятные породы.

In [17]:
from PIL import Image, ImageOps

@torch.inference_mode()
def predict_image(
    image_path,
    model,
    transform,
    class_names,
    top_k=3
):
  if not 1 <= top_k <= len(class_names):
    raise ValueError (
        "top_k должен быть от 1 до числа классов"
    )

  model.eval()
  model_device = next(model.parameters()).device

  with Image.open(image_path) as source:
    image = ImageOps.exif_transpose(source).convert("RGB")

  inputs = transform(image).unsqueeze(0).to(model_device)

  logits = model(inputs)
  probabilities = torch.softmax(logits, dim=1)

  top_probabilities, top_indices = probabilities.topk(
      top_k,
      dim=1
  )

  return [
      {
          "class_name": class_names[index],
          "probability": probability,
      }
      for index, probability in zip(
          top_indices[0].cpu().tolist(),
          top_probabilities[0].cpu().tolist()
      )
  ]


Здесь `softmax` преобразует logits в распределение по классам, а `top_k` выбирает самые большие значения и их индексы

### Автономный пример

Чтобы notebook выполнялся сверху вниз даже без загруженной вручную фотографии, сначала проверим inference на одном исходном изображении из test-выборки.

In [ ]:
raw_test_dataset = OxfordIIITPet(
    root="data",
    split="test",
    target_types="category",
    download=False,
)

demo_image, demo_label = raw_test_dataset[0]
demo_image_path = Path("demo_inference.jpg")
demo_image.save(demo_image_path)

demo_predictions = predict_image(
    image_path=demo_image_path,
    model=aug_fine_tune_model,
    transform=resnet_val_transform,
    class_names=checkpoint["class_names"],
)

print("True class:", raw_test_dataset.classes[demo_label])
print("Top-3 predictions:")

for prediction in demo_predictions:
    print(
        f"{prediction['class_name']}: "
        f"{prediction['probability']:.2%}"
    )

### Своя фотография

Ниже остаётся пример с пользовательским изображением. Если файла нет в текущей среде Colab, ячейка не завершится ошибкой и подскажет, что нужно загрузить.

In [18]:
custom_image_path = Path("/content/Abyssinian_126.jpg")

if custom_image_path.exists():
    predictions = predict_image(
        image_path=custom_image_path,
        model=aug_fine_tune_model,
        transform=resnet_val_transform,
        class_names=checkpoint["class_names"],
    )

    for prediction in predictions:
        print(
            f"{prediction['class_name']}: "
            f"{prediction['probability']:.2%}"
        )
else:
    print(
        "Пользовательская фотография не найдена. "
        "Загрузите файл в Colab и измените custom_image_path, затем повторно выполните эту ячейку."
    )


Abyssinian: 84.79%
Russian Blue: 12.43%
British Shorthair: 2.63%
